# Benchmark - run a two-player grid on Colab
Plays the two-player games of a grid (`grids/duel_*.yaml`: chess, Connect Four) that have not been played yet, one model at a time, and saves one record per game on Google Drive, next to the records of the textual games.
The agent is the same of the textual games (same modes), with the prompts of `duel_prompts.py`; the levels of a game are its opponents (`random`, `greedy`, Stockfish `sf<N>` / `elo<N>`, or the model itself `self-<mode>`, see `duel.py`).
If the run stops (or the runtime is deleted), run the notebook again: it resumes from the games not yet saved.

Recommended order: a pilot (`limit = 8`) to see the speed and the context sizes, then `duel_tier1.yaml`, then `duel_selfplay.yaml` (self-play, neutral prompt) and `duel_free.yaml` (the same self-play games with the free prompt, which does not structure the reasoning).

# Setup
Clones the repository (only the benchmark folder: the two-player games need no game files), installs vLLM, python-chess and Stockfish, and moves into `Code/Benchmark`. Locally it does nothing.
The first time on a new Colab machine the installation takes a few minutes: restart the kernel after it (and then run all the cells), because it can update torch.

In [1]:
import os
import sys
import shutil
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/UPROOT01/LLMs-Play-Textual-Games.git"
    repo_dir = "/content/LLMs-Play-Textual-Games"
    if not os.path.isfile(f"{repo_dir}/Code/Benchmark/run.py"):
        shutil.rmtree(repo_dir, ignore_errors=True) # leftover of a failed clone, or a clone without the benchmark
        # sparse clone: only the folders needed (the whole Games folder is >1 GB)
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", repo_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", repo_dir, "sparse-checkout", "add", "Code/Benchmark"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.31.0", "pyyaml", "chess"], check=True)
    subprocess.run(["apt-get", "install", "-qq", "-y", "stockfish"], check=True) # the sf<N> / elo<N> opponents
    os.chdir(f"{repo_dir}/Code/Benchmark")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

Working directory: /content/LLMs-Play-Textual-Games/Code/Benchmark


In [3]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import torch

assert torch.cuda.is_available(), "No GPU available: connect to a Colab runtime with a GPU (e.g. L4)"
! nvidia-smi

Tue Oct 27 11:24:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   42C    P8             13W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# What to run
- `grid`: the grid file (`grids/duel_*.yaml`). `models`: only these models of the grid (`None`: all of them, one after the other).
- `limit`: at most this many games per model (`None`: all). Use it for a pilot.
- `n_parallel`: games played at the same time. vLLM answers them together: more games, more tokens per minute, until the GPU memory is full.
  `"auto"` decides by itself: it starts a new game only if the contexts of all the games, at the size they are expected to reach, fit in the GPU memory (see `engine.ContextBudget`).
  The status line shows how full the GPU memory is expected to be. A number (e.g. `8`) plays always that many.

To change the models or their vLLM settings (e.g. a smaller `max_model_len` if a model does not fit), see `profiles.py`.

In [12]:
from records import LEGACY_RESULTS
from run import summary
from run_duel import load_duel_grid, run_duel_grid

grid = load_duel_grid("grids/duel_free.yaml") # or duel_plain.yaml (no reasoning instruction), duel_selfplay.yaml (neutral prompt), duel_tier1.yaml (bots)
models = None # None: all the models of the grid, or e.g. ["Qwen/Qwen3-4B"]
limit = 8 # 8: the pilot (seed 1, the same 8 games of the neutral pilot); None: the whole grid
n_parallel = "auto"

if IN_COLAB:
    assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not mounted: run the cell above"
    results = "/content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results"
else:
    results = "results_new"
read = [LEGACY_RESULTS] # the old experiments: their games are not played again
print("Results folder:", results)
summary(grid, results, read, models)

Results folder: /content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results
model                                          mode  played  left  ~Mtokens left
Qwen/Qwen3-4B                                    e1       0    40               
Qwen/Qwen3-4B                                    r4       0    40            0.4
Qwen/Qwen3-4B                                   er4       0    40            0.7
Qwen/Qwen3-4B                                   lr4       0    40            0.3
total: 0 played, 160 left (the token estimate uses the games already played with the same mode, any model)


# Run
Every game is saved as soon as it ends. Stopping the cell is safe: the games being played are lost, the others are kept.
If loading the second model of a run fails because the GPU memory is full, restart the runtime and run everything again: the first model's games are skipped.

In [13]:
run_duel_grid(grid, results, read, models=models, limit=limit, n_parallel=n_parallel)

Qwen/Qwen3-4B: 8 games to play


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


[rank0]:W1027 11:35:58.080000 1030 torch/_inductor/utils.py:1953] Not enough SMs to use max_autotune_gemm mode
Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00, 13.88it/s]
2026-10-27 11:36:50,743 - INFO - core.py:564 - flashinfer.jit: Building JIT module sampling; this can take several minutes on first use.
Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:02<00:00, 15.41it/s]


Qwen/Qwen3-4B: KV cache of 78816 tokens


1 games running, 0 waiting, 7 ended; since the start 6.8 games/hour, 3779 tokens/min; last 10 min 6.0 games/hour, 2059 tokens/min; contexts 32% of the KV cache at their expected end

('Qwen/Qwen3-4B', 'chess', 'neutral', 'e1', 'self-e1', 1): steps 8, score 0.5/1, 1069 s
('Qwen/Qwen3-4B', 'chess', 'neutral', 'r4', 'self-e1', 1): steps 16, score 0.5/1, 1373 s
('Qwen/Qwen3-4B', 'chess', 'neutral', 'lr4', 'self-e1', 1): steps 15, score 0.5/1, 1280 s
('Qwen/Qwen3-4B', 'chess', 'neutral', 'r4', 'self-r4', 1): steps 30, score 0.5/1, 303 s
('Qwen/Qwen3-4B', 'chess', 'neutral', 'er4', 'self-e1', 1): steps 47, score 0.5/1, 2901 s
('Qwen/Qwen3-4B', 'chess', 'neutral', 'er4', 'self-r4', 1): steps 8, score 0.5/1, 338 s
('Qwen/Qwen3-4B', 'chess', 'neutral', 'lr4', 'self-r4', 1): steps 19, score 0.5/1, 350 s
('Qwen/Qwen3-4B', 'chess', 'neutral', 'e1', 'self-r4', 1): steps 31, score 0.5/1, 2327 s
Qwen/Qwen3-4B: done in 62 min


# Quick look
Per model, game, opponent and mode: average score on the board and adjudicated (win 1, draw 0.5, loss 0: the chess games stopped at `max_steps` are decided by Stockfish's evaluation of the final position, see `duel.py`), wins, draws and losses after the adjudication, games stopped at `max_steps`, illegal answers and random moves played for the agent per game, thousands of generated tokens per game.

In [14]:
from records import load_table

table = load_table(results)
games = table[(table.source == "benchmark-duel") & table.game.isin(grid["games"]) & table.model.isin(grid["models"])
              & table["mode"].isin(grid["modes"]) & table.seed.isin(grid["seeds"])].copy()
# records played before the adjudication existed: the result on the board
games["adjudicated_result"] = games.get("adjudicated_result", games.result).fillna(games.result)
games["adjudicated_score"] = games.adjudicated_result.map({"win": 1, "draw": 0.5, "loss": 0})
games.groupby(["model", "game", "opponent", "mode"]).agg(
    score=("score", "mean"), adjudicated=("adjudicated_score", "mean"),
    wins=("adjudicated_result", lambda r: (r == "win").sum()), draws=("adjudicated_result", lambda r: (r == "draw").sum()),
    losses=("adjudicated_result", lambda r: (r == "loss").sum()), stopped=("termination", lambda t: (t == "max_steps").sum()),
    illegal=("illegal", "mean"), forced=("forced", "mean"),
    ktokens=("tokens_generated", lambda t: t.mean() / 1000)).round(2)

score  adjudicated  wins  draws  losses  \
model         game  opponent mode                                            
Qwen/Qwen3-4B chess self-e1  e1      0.5          0.5     0      1       0   
                             er4     0.5          0.5     0      1       0   
                             lr4     0.5          0.5     0      1       0   
                             r4      0.5          0.5     0      1       0   
                    self-r4  e1      0.5          0.5     0      1       0   
                             er4     0.5          0.5     0      1       0   
                             lr4     0.5          0.5     0      1       0   
                             r4      0.5          0.5     0      1       0   

                                   stopped  illegal  forced  ktokens  
model         game  opponent mode                                     
Qwen/Qwen3-4B chess self-e1  e1          0      0.0     0.0    11.28  
                             er4         0      3.0     0.0    16.75  
                             lr4         0      1.0     0.0     2.94  
                             r4          0      1.0     0.0     2.51  
                    self-r4  e1          0      1.0     0.0    45.97  
                             er4         0      0.0     0.0     3.88  
                             lr4         0      1.0     0.0     4.12  
                             r4          0      5.0     0.0     2.22